In [ ]:
import copy
from typing import Callable
import time

import duckdb
import faiss
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

from retriever_workflow import train_val_split, \
                        get_unique_movies, get_unique_users, train_retriever
from two_towers import UserTower, MovieTower, TwoTowerModel
from recall_at_k import recall_at_ks_macro, get_user_embeddings, get_movie_embeddings
from retriever_loss import info_nce, build_interaction_matrix
from recommender_datasets import PositiveRatings

In [2]:
device_name = "cpu"

In [3]:
with duckdb.connect("/home/maria-khanina/mkgit/movies_project/data/database.duckdb", read_only=True) as db_conn:
    ratings_df = db_conn.execute("""
        SELECT *
        FROM ratings
            INNER JOIN movies USING (movie_id)
            INNER JOIN users USING (user_id)
    """).df()

In [4]:
unique_movies = ratings_df[["movie_id", "genres"]].drop_duplicates()
genres_encoded = unique_movies["genres"].str.get_dummies(sep="|")
genres_cols = genres_encoded.columns.to_list()
genres_num = len(genres_cols)

movies_with_genres = pd.concat(
    [unique_movies[["movie_id"]], genres_encoded], 
    axis=1,
)

ratings_df = ratings_df.merge(
    movies_with_genres, 
    on="movie_id", 
    how="left",
)

In [5]:
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

cat_cols = ["gender"]

ratings_df[cat_cols] = encoder.fit_transform(ratings_df[cat_cols]) + 1
ratings_df["occupation"] = ratings_df["occupation"] + 1

In [6]:
max_user = int(ratings_df["user_id"].max())
max_movie = int(ratings_df["movie_id"].max())
max_age = int(ratings_df["age"].max())
max_gender = int(ratings_df["gender"].max())
max_job = int(ratings_df["occupation"].max())

In [7]:
cold_user_ratio = 0.05

In [8]:
train_all_df, test_df = train_val_split(
    ratings_df,
    user_col="user_id",
    time_col="timestamp",
    cold_user_ratio=cold_user_ratio,
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

train_big_df, outer_val_df = train_val_split(
    train_all_df,
    user_col="user_id",
    time_col="timestamp",
    cold_user_ratio=cold_user_ratio,
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

train_df, val_df = train_val_split(
    train_big_df,
    user_col="user_id",
    time_col="timestamp",
    cold_user_ratio=cold_user_ratio,
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

In [9]:
movie_counts = train_df["movie_id"].value_counts().sort_index()

popularity = torch.ones(
    max_movie + 1, 
    dtype=torch.float32,
)

popularity[movie_counts.index.values] += torch.tensor(
    movie_counts.values, 
    dtype=torch.float32,
)

frequencies = popularity / popularity.sum()
movie_log_freqs = torch.log(frequencies)

device = torch.device(device_name)
movie_log_freqs = movie_log_freqs.to(device)

In [10]:
var_key_map = {
    "user_ids": "user_id",
    "movie_ids": "movie_id",
    "ages": "age",
    "genders": "gender",
    "jobs": "job",
    "genres": "genre",
}

In [11]:
train_dataset = PositiveRatings(
    train_df, 
    genres_cols,
    min_rating=4,
    is_training=True,
    user_dropout=cold_user_ratio,
)

trained_users = torch.unique(train_dataset.user_ids)

val_dataset = PositiveRatings(
    val_df, 
    genres_cols, 
    trained_users=trained_users,
)
outer_val_dataset = PositiveRatings(
    outer_val_df, 
    genres_cols,
    trained_users=trained_users,
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=2048,
    shuffle=True,
    num_workers=4,
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=2048,
    shuffle=False,
    num_workers=4,
)

outer_val_loader = DataLoader(
    outer_val_dataset, 
    batch_size=2048,
    shuffle=False,
    num_workers=4,
)

In [12]:
train_users = train_df["user_id"].unique()
outer_val_users = outer_val_df["user_id"].unique()
val_users = val_df["user_id"].unique()

outer_cold_user_ids = torch.tensor(
    np.setdiff1d(outer_val_users, train_users),
    dtype=torch.long,
)

cold_user_ids = torch.tensor(
    np.setdiff1d(val_users, train_users),
    dtype=torch.long,
)

In [13]:
interactions = build_interaction_matrix(
    user_indices=train_dataset.user_ids,
    movie_indices=train_dataset.movie_ids,
    max_user_id=max_user,
    max_movie_id=max_movie,
)

In [14]:
user_tower = UserTower(
    users_num=max_user,
    ages_num=max_age,
    genders_num=max_gender,
    jobs_num=max_job,
    user_dim=32,
    age_dim=4,
    gender_dim=2,
    job_dim=8,
    output_dim=64,
    hidden_dims=[256, 128],
    dropout_ratio=0.1,
)
    
movie_tower = MovieTower(
    movies_num=max_movie,
    genres_num=genres_num,
    movie_dim=32,
    genre_dim=8,
    output_dim=64,
    hidden_dims=[256, 128],
    dropout_ratio=0.1,
)

retriever = TwoTowerModel(
    user_tower=user_tower,
    movie_tower=movie_tower,
    normalize_outputs=True,
)

In [25]:
movie_ids, movie_genres = get_unique_movies(
    ratings_df,
    genres_cols,
)

In [16]:
train_retriever(
    model=retriever,
    train_loader=train_loader,
    val_loader=val_loader,
    all_movie_ids=movie_ids,
    all_genres=movie_genres,
    cold_user_ids=cold_user_ids,
    var_key_map=var_key_map,
    embedding_loss=info_nce,
    epochs=10_000,
    learning_rate=0.002,
    patience=30,
    tol=0.0001,
    weight_decay=0.0001,
    device_name=device_name,
    recall_k=100,
    interaction_matrix=interactions,
    check_interactions=True,
    temperature=0.05,
    movie_log_freqs=movie_log_freqs,
)

Epoch 1/10000, Train Loss: 7.9652, Val Loss: 7.7697.  Val recall@100-macro: 0.2740
Epoch 2/10000, Train Loss: 7.4694, Val Loss: 7.6045.  Val recall@100-macro: 0.2961
Epoch 3/10000, Train Loss: 7.3228, Val Loss: 7.5395.  Val recall@100-macro: 0.3072
Epoch 4/10000, Train Loss: 7.2011, Val Loss: 7.4855.  Val recall@100-macro: 0.3255
Epoch 5/10000, Train Loss: 7.0995, Val Loss: 7.4310.  Val recall@100-macro: 0.3468
Epoch 6/10000, Train Loss: 7.0158, Val Loss: 7.3852.  Val recall@100-macro: 0.3619
Epoch 7/10000, Train Loss: 6.9484, Val Loss: 7.3544.  Val recall@100-macro: 0.3757
Epoch 8/10000, Train Loss: 6.8960, Val Loss: 7.3378.  Val recall@100-macro: 0.3874
Epoch 9/10000, Train Loss: 6.8537, Val Loss: 7.3170.  Val recall@100-macro: 0.3932
Epoch 10/10000, Train Loss: 6.8200, Val Loss: 7.2995.  Val recall@100-macro: 0.3979
Epoch 11/10000, Train Loss: 6.7917, Val Loss: 7.2870.  Val recall@100-macro: 0.3995
Epoch 12/10000, Train Loss: 6.7674, Val Loss: 7.2802.  Val recall@100-macro: 0.4071
E

In [26]:
ks = [10, 50, 100, 200, 500]

recall_at_ks_macro(
    retriever,
    outer_val_loader,
    var_key_map,
    movie_ids,
    movie_genres,
    torch.device("cpu"),
    outer_cold_user_ids,
    ks=ks,
)

{10: {'global': 0.07671014219522476,
  'warm': 0.07873024046421051,
  'cold': 0.0581376776099205},
 50: {'global': 0.2816634178161621,
  'warm': 0.2915629744529724,
  'cold': 0.1906481683254242},
 100: {'global': 0.4266993999481201,
  'warm': 0.4397991895675659,
  'cold': 0.3062618374824524},
 200: {'global': 0.591697096824646,
  'warm': 0.6075193285942078,
  'cold': 0.44623008370399475},
 500: {'global': 0.8022875785827637,
  'warm': 0.8159952163696289,
  'cold': 0.6762605905532837}}

In [18]:
def build_and_search_ann(
    index_string: str,
    vector_dim: int,
    movie_vectors: np.ndarray,
    movie_ids: np.ndarray,
    user_vectors: np.ndarray,
    n_neighbors=500,
    nprobe: int = 10,
    efSearch=32,
) -> tuple[np.ndarray, np.ndarray, float, int]:
    
    base_index = faiss.index_factory(
        vector_dim, 
        index_string, 
        faiss.METRIC_INNER_PRODUCT,
    )
    index = faiss.IndexIDMap(base_index)
    
    if not index.is_trained:
        index.train(movie_vectors)
        
    index.add_with_ids(movie_vectors, movie_ids)
    if "IVF" in index_string:
        faiss.ParameterSpace().set_index_parameter(index, "nprobe", nprobe)
        
    elif "HNSW" in index_string:
        faiss.ParameterSpace().set_index_parameter(index, "efSearch", efSearch)
    
    index_ram_bytes = faiss.serialize_index(index).nbytes
    
    start_time = time.perf_counter()
    _, indices = index.search(user_vectors, n_neighbors)
    search_time = time.perf_counter() - start_time

    return indices, search_time, index_ram_bytes


def calculate_metrics(
    exact_idx: np.ndarray,
    approx_idx: np.ndarray,
) -> float:
    
    n_users = exact_idx.shape[0]
    n_neighbors = exact_idx.shape[1]
    
    matches = (exact_idx[:, :, np.newaxis] 
                == approx_idx[:, np.newaxis, :])
    total_overlap = matches.sum()
        
    return total_overlap / (n_users * n_neighbors)

In [19]:
movie_ids, genres = get_unique_movies(ratings_df, genres_cols)
user_ids, ages, genders, jobs = get_unique_users(ratings_df)

movie_ids = movie_ids.to(device)
genres = genres.to(device)

user_ids = user_ids.to(device)
ages = ages.to(device)
genders = genders.to(device)
jobs = jobs.to(device)

retriever.eval()
with torch.no_grad():
    movie_vectors = get_movie_embeddings(
        retriever, 
        movie_ids, 
        genres
    )
    
    user_vectors = get_user_embeddings(
        retriever, 
        user_ids, 
        ages, 
        genders, 
        jobs
    )

movie_vectors_np = movie_vectors.cpu().numpy().astype(np.float32)
movie_ids = movie_ids.cpu().numpy().astype(np.int64)

user_vectors_np = user_vectors.cpu().numpy().astype(np.float32)

In [20]:
def report_ann(
    exact_idx: np.ndarray,
    exact_time: int,
    index_str: str,
    report_df: pd.DataFrame,
    vector_dim=64,
    n_neighbors=50,
    nprobe=10,
    efSearch=64,
):
    approx_idx, search_time, ram = build_and_search_ann(
        index_string=index_str,
        vector_dim=vector_dim,
        movie_vectors=movie_vectors_np,
        movie_ids=movie_ids,
        user_vectors=user_vectors_np,
        n_neighbors=n_neighbors,
        nprobe=nprobe,
        efSearch=efSearch,
    )

    if "IVF" in index_str:
        parameters = f"nprobe: {nprobe}"
    elif "HNSW" in index_str:
        parameters = f"efSearch: {efSearch}"
    else:
        parameters = "-"

    report_df.loc[len(report_df)] = {
        "Index": index_str,
        "RAM (MB)": ram / (1024 * 1024),
        "Search time (seconds)": search_time,
        "Speedup (x)": exact_time / search_time,
        "Recall": calculate_metrics(exact_idx, approx_idx),
        "Parameters": parameters
    }



In [21]:
exact_idx, exact_time, exact_ram = build_and_search_ann(
    index_string="Flat",
    vector_dim=64,
    movie_vectors=movie_vectors_np,
    movie_ids=movie_ids,
    user_vectors=user_vectors_np,
    n_neighbors=50,
)

report_df = pd.DataFrame([{
    "Index": "Flat",
    "RAM (MB)": exact_ram / (1024 * 1024),
    "Search time (seconds)": exact_time,
    "Speedup (x)": 1.0,
    "Recall": 1.0,
    "Parameters": "-",
}])

In [22]:
report_ann(
    exact_idx,
    exact_time,
    f"HNSW32,PQ8x4",
    report_df,
)

report_ann(
    exact_idx,
    exact_time,
    f"HNSW32",
    report_df,
)

report_ann(
    exact_idx,
    exact_time,
    f"HNSW64,PQ32x6",
    report_df,
)

report_ann(
    exact_idx,
    exact_time,
    f"HNSW64",
    report_df,
)

report_ann(
    exact_idx,
    exact_time,
    f"HNSW128",
    report_df,
)


report_ann(
    exact_idx,
    exact_time,
    f"IVF64,Flat",
    report_df,
    nprobe=10,
)

report_ann(
    exact_idx,
    exact_time,
    f"IVF32,Flat",
    report_df,
    nprobe=5,
)

report_ann(
    exact_idx,
    exact_time,
    f"IVF16,Flat",
    report_df,
    nprobe=2,
)

report_ann(
    exact_idx,
    exact_time,
    f"IVF8,Flat",
    report_df,
    nprobe=8,
)

In [23]:
report_df.sort_values(by="Recall", ascending=False)

,Index,RAM (MB),Search time (seconds),Speedup (x),Recall,Parameters
0,Flat,0.933146,0.149853,1.000000,1.000000,-
9,"IVF8,Flat",0.963524,0.035552,4.215087,1.000000,nprobe: 8
4,HNSW64,2.798961,0.044741,3.349371,0.999368,efSearch: 64
5,HNSW128,4.605113,0.035374,4.236311,0.999298,efSearch: 64
2,HNSW32,1.892600,0.027250,5.499292,0.998364,efSearch: 64
6,"IVF64,Flat",0.977623,0.012507,11.981286,0.921805,nprobe: 10
7,"IVF32,Flat",0.969566,0.022741,6.589687,0.862752,nprobe: 5
3,"HNSW64,PQ32x6",1.994663,0.082346,1.819806,0.796318,efSearch: 64
8,"IVF16,Flat",0.965538,0.011171,13.414337,0.698550,nprobe: 2
1,"HNSW32,PQ8x4",1.005898,0.050518,2.966340,0.390265,efSearch: 64
